In [ ]:
# ============================================================
# SECTION 1: Setup & Data Discovery
# ============================================================

import os, re, gc
from glob import glob
from collections import defaultdict
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# ---------- CONFIG ----------
DATASET_ROOT = "/kaggle/input/nbaiot-dataset"
OUT_RESULTS = "/kaggle/working/results"
OUT_CLIENTS = "/kaggle/working/fl_clients"

RANDOM_STATE = 42
TEST_SIZE = 0.20
VAL_SIZE = 0.10  # NEW: validation set for proper ML

os.makedirs(OUT_RESULTS, exist_ok=True)
os.makedirs(OUT_CLIENTS, exist_ok=True)

print("=" * 60)
print("SECTION 1: Data Discovery")
print("=" * 60)

# ---------- Discover all CSVs ----------
all_csvs = sorted(glob(os.path.join(DATASET_ROOT, "*.csv")))
print(f"\nTotal CSV files found: {len(all_csvs)}")

# ---------- Device name mapping (based on N-BaIoT structure) ----------
# The dataset has numeric prefixes that map to device names
DEVICE_NAME_MAP = {
    '1': 'Danmini_Doorbell',
    '2': 'Ecobee_Thermostat',
    '3': 'Ennio_Doorbell',
    '4': 'Philips_B120N10_Baby_Monitor',
    '5': 'Provision_PT_737E_Security_Camera',
    '6': 'Provision_PT_838_Security_Camera',
    '7': 'Samsung_SNH_1011_N_Webcam',
    '8': 'SimpleHome_XCS7_1002_WHT_Security_Camera',
    '9': 'SimpleHome_XCS7_1003_WHT_Security_Camera'
}

# ---------- Separate device files from metadata ----------
device_files = []
meta_files = []

for csv_path in all_csvs:
    basename = os.path.basename(csv_path)
    # Device CSVs start with number + dot (e.g., "1.benign.csv", "5.mirai.scan.csv")
    if re.match(r'^\d+\.', basename):
        device_files.append(csv_path)
    else:
        meta_files.append(csv_path)

print(f"\nDevice CSV files: {len(device_files)}")
print(f"Metadata/other files: {len(meta_files)}")

if meta_files:
    print("\nIgnored files (not device data):")
    for mf in meta_files[:10]:
        print(f"  - {os.path.basename(mf)}")

# ---------- Group files by device ----------
device_groups = defaultdict(list)

for filepath in device_files:
    basename = os.path.basename(filepath)
    device_num = basename.split(".")[0]  # Extract '1', '2', etc.
    device_groups[device_num].append(filepath)

# Sort by numeric order
device_nums = sorted(device_groups.keys(), key=lambda x: int(x))

print(f"\n{'Device ID':<12} {'Device Name':<45} {'Files'}")
print("-" * 80)
for device_num in device_nums:
    device_name = DEVICE_NAME_MAP.get(device_num, f"Unknown_Device_{device_num}")
    file_count = len(device_groups[device_num])
    print(f"{device_num:<12} {device_name:<45} {file_count}")

print("\n✅ SECTION 1 COMPLETE")
print(f"Found {len(device_nums)} devices with {len(device_files)} total files")
print("-" * 80)

In [ ]:
# ============================================================
# SECTION 2: Data Loading & Train/Val/Test Split (NO LEAKAGE)
# ============================================================

print("\n" + "=" * 60)
print("SECTION 2: Data Loading & Clean Train/Val/Test Split")
print("=" * 60)

device_data_summary = []
MIN_SAMPLES_FOR_SPLIT = 10  # Minimum samples needed to do stratified split

for device_num in device_nums:
    device_name = DEVICE_NAME_MAP.get(device_num, f"Unknown_{device_num}")
    files = sorted(device_groups[device_num])
    
    print(f"\n{'─' * 60}")
    print(f"📱 Device {device_num}: {device_name}")
    print(f"{'─' * 60}")
    
    # ---------- Load all files for this device ----------
    dfs = []
    for filepath in files:
        basename = os.path.basename(filepath)
        try:
            df = pd.read_csv(filepath)
            # Label: 'benign' in filename → 0, else → 1
            label = 0 if 'benign' in basename.lower() else 1
            df['Attack_label'] = label
            dfs.append(df)
            print(f"  ✓ Loaded: {basename:<50} ({len(df):>8} rows, label={label})")
        except Exception as e:
            print(f"  ✗ ERROR loading {basename}: {e}")
            continue
    
    if len(dfs) == 0:
        print(f"  ⚠️  No valid data for device {device_num}, skipping")
        continue
    
    # ---------- Concatenate all data for this device ----------
    df_device = pd.concat(dfs, ignore_index=True)
    del dfs
    gc.collect()
    
    # ---------- Data cleaning ----------
    # Convert to numeric, fill NaN with 0
    df_device = df_device.apply(pd.to_numeric, errors='coerce').fillna(0)
    
    # Drop non-feature columns if present
    non_feature_cols = ['timestamp', 'flow_id', 'proto', 'service', 'label']
    cols_to_drop = [c for c in non_feature_cols if c in df_device.columns]
    if cols_to_drop:
        df_device = df_device.drop(columns=cols_to_drop)
    
    # Ensure Attack_label exists and is integer
    if 'Attack_label' not in df_device.columns:
        df_device['Attack_label'] = 0
    df_device['Attack_label'] = df_device['Attack_label'].astype(int)
    
    # ---------- Separate features and labels ----------
    X = df_device.drop(columns=['Attack_label'], errors='ignore').values
    y = df_device['Attack_label'].values
    
    n_total = len(y)
    n_malicious = int(y.sum())
    n_benign = n_total - n_malicious
    
    print(f"\n  Total samples: {n_total:,}")
    print(f"  Benign: {n_benign:,} ({n_benign/n_total*100:.1f}%)")
    print(f"  Malicious: {n_malicious:,} ({n_malicious/n_total*100:.1f}%)")
    
    # ---------- CRITICAL: Proper 3-way split ----------
    # Train: 70%, Val: 10%, Test: 20%
    
    if n_total < MIN_SAMPLES_FOR_SPLIT:
        print(f"  ⚠️  Too few samples ({n_total}), putting all in train")
        X_train, y_train = X, y
        X_val, y_val = np.empty((0, X.shape[1])), np.empty(0, dtype=int)
        X_test, y_test = np.empty((0, X.shape[1])), np.empty(0, dtype=int)
    
    elif n_benign == 0 or n_malicious == 0:
        print(f"  ⚠️  Single-class data, putting all in train")
        X_train, y_train = X, y
        X_val, y_val = np.empty((0, X.shape[1])), np.empty(0, dtype=int)
        X_test, y_test = np.empty((0, X.shape[1])), np.empty(0, dtype=int)
    
    else:
        # First split: 80% train+val, 20% test
        X_trainval, X_test, y_trainval, y_test = train_test_split(
            X, y, 
            test_size=TEST_SIZE, 
            random_state=RANDOM_STATE, 
            stratify=y
        )
        
        # Second split: from the 80%, take 12.5% as val (which is 10% of total)
        # 12.5% of 80% = 10% of 100%
        val_from_trainval = VAL_SIZE / (1 - TEST_SIZE)  # 0.125
        
        if len(y_trainval) >= MIN_SAMPLES_FOR_SPLIT and min(y_trainval.sum(), len(y_trainval) - y_trainval.sum()) >= 2:
            X_train, X_val, y_train, y_val = train_test_split(
                X_trainval, y_trainval,
                test_size=val_from_trainval,
                random_state=RANDOM_STATE,
                stratify=y_trainval
            )
        else:
            X_train, y_train = X_trainval, y_trainval
            X_val, y_val = np.empty((0, X.shape[1])), np.empty(0, dtype=int)
    
    print(f"\n  Split sizes:")
    print(f"    Train: {len(y_train):,} ({len(y_train)/n_total*100:.1f}%) - malicious: {y_train.sum()}")
    print(f"    Val:   {len(y_val):,} ({len(y_val)/n_total*100:.1f}%) - malicious: {y_val.sum()}")
    print(f"    Test:  {len(y_test):,} ({len(y_test)/n_total*100:.1f}%) - malicious: {y_test.sum()}")
    
    # ---------- Save raw (unscaled) splits ----------
    device_dir = os.path.join(OUT_CLIENTS, f"device_{device_num}_{device_name}")
    os.makedirs(device_dir, exist_ok=True)
    
    # Save as compressed npz
    np.savez_compressed(
        os.path.join(device_dir, "train_raw.npz"),
        X=X_train.astype(np.float32),
        y=y_train.astype(np.int8)
    )
    np.savez_compressed(
        os.path.join(device_dir, "val_raw.npz"),
        X=X_val.astype(np.float32),
        y=y_val.astype(np.int8)
    )
    np.savez_compressed(
        os.path.join(device_dir, "test_raw.npz"),
        X=X_test.astype(np.float32),
        y=y_test.astype(np.int8)
    )
    
    print(f"  ✅ Saved raw splits to: {device_dir}")
    
    # ---------- Store summary ----------
    device_data_summary.append({
        'device_num': device_num,
        'device_name': device_name,
        'n_total': n_total,
        'n_train': len(y_train),
        'n_val': len(y_val),
        'n_test': len(y_test),
        'n_benign': n_benign,
        'n_malicious': n_malicious,
        'train_malicious': int(y_train.sum()),
        'val_malicious': int(y_val.sum()),
        'test_malicious': int(y_test.sum())
    })
    
    # Clean up
    del df_device, X, y, X_train, X_val, X_test, y_train, y_val, y_test
    gc.collect()

# ---------- Save summary ----------
summary_df = pd.DataFrame(device_data_summary)
summary_path = os.path.join(OUT_RESULTS, "data_split_summary.csv")
summary_df.to_csv(summary_path, index=False)

print("\n" + "=" * 60)
print("✅ SECTION 2 COMPLETE")
print("=" * 60)
print(f"\nData summary saved to: {summary_path}")
print("\nPer-device summary:")
print(summary_df.to_string(index=False))
print("\n" + "=" * 60)

In [ ]:
# ============================================================
# SECTION 3: Scaler Fitting & Signature Construction (TRAIN ONLY)
# ============================================================

print("\n" + "=" * 60)
print("SECTION 3: Normalization & Signature Extraction")
print("=" * 60)

print("\n⚠️ CRITICAL: Using TRAINING DATA ONLY for:")
print(" - Fitting scalers")
print(" - Computing device signatures")
print(" - PCA transformation")
print("=" * 60)

N_PCA = 5  # Number of PCA components for behavioral embeddings

device_signatures = []
signature_metadata = []

for device_num in device_nums:
    device_name = DEVICE_NAME_MAP.get(device_num, f"Unknown_{device_num}")
    device_dir = os.path.join(OUT_CLIENTS, f"device_{device_num}_{device_name}")

    print(f"\n{'─' * 60}")
    print(f"📱 Device {device_num}: {device_name}")
    print(f"{'─' * 60}")

    # ---------- Load RAW splits ----------
    train_path = os.path.join(device_dir, "train_raw.npz")
    val_path   = os.path.join(device_dir, "val_raw.npz")
    test_path  = os.path.join(device_dir, "test_raw.npz")

    train_data = np.load(train_path)
    val_data   = np.load(val_path)
    test_data  = np.load(test_path)

    X_train, y_train = train_data["X"], train_data["y"]
    X_val,   y_val   = val_data["X"],   val_data["y"]
    X_test,  y_test  = test_data["X"],  test_data["y"]

    print(f" Loaded: Train={X_train.shape}, Val={X_val.shape}, Test={X_test.shape}")

    if X_train.shape[0] == 0:
        print(f" ⚠️ No training data, skipping device {device_num}")
        continue

    # ---------- FIT SCALER (TRAIN ONLY) ----------
    scaler = StandardScaler()
    scaler.fit(X_train)

    mean_vec = scaler.mean_.astype(np.float32)
    scale_vec = scaler.scale_.astype(np.float32)

    # Avoid division by zero
    scale_safe = np.where(scale_vec == 0.0, 1.0, scale_vec)

    np.savez_compressed(
        os.path.join(device_dir, "scaler_params.npz"),
        mean=mean_vec,
        scale=scale_safe
    )

    print(" ✓ Fitted scaler on training data")
    print(f"   Mean range : [{mean_vec.min():.4f}, {mean_vec.max():.4f}]")
    print(f"   Scale range: [{scale_safe.min():.4f}, {scale_safe.max():.4f}]")

    # ---------- APPLY SCALER ----------
    X_train_scaled = (X_train - mean_vec) / scale_safe
    X_val_scaled   = (X_val   - mean_vec) / scale_safe if X_val.size > 0 else X_val
    X_test_scaled  = (X_test  - mean_vec) / scale_safe if X_test.size > 0 else X_test

    np.savez_compressed(
        os.path.join(device_dir, "train_scaled.npz"),
        X=X_train_scaled.astype(np.float32),
        y=y_train.astype(np.int8)
    )
    np.savez_compressed(
        os.path.join(device_dir, "val_scaled.npz"),
        X=X_val_scaled.astype(np.float32),
        y=y_val.astype(np.int8)
    )
    np.savez_compressed(
        os.path.join(device_dir, "test_scaled.npz"),
        X=X_test_scaled.astype(np.float32),
        y=y_test.astype(np.int8)
    )

    print(" ✓ Saved scaled train/val/test splits")

    # ---------- COMPUTE DEVICE SIGNATURE (TRAIN ONLY) ----------
    print("\n Computing signature from TRAINING data only:")

    mean_features = np.mean(X_train_scaled, axis=0)
    std_features  = np.std(X_train_scaled, axis=0)

    attack_ratio = float(np.mean(y_train))
    print(f" Attack ratio (train): {attack_ratio:.4f}")

    # ---------- PCA (TRAIN ONLY, NON-ZERO VAR FEATURES) ----------
    variance_mask = std_features > 1e-12
    n_valid_feats = int(variance_mask.sum())

    pca_embeddings = np.zeros(N_PCA, dtype=np.float32)

    if n_valid_feats > 0 and X_train_scaled.shape[0] > 1:
        n_components = min(N_PCA, n_valid_feats, X_train_scaled.shape[0] - 1)

        if n_components > 0:
            try:
                pca = PCA(
                    n_components=n_components,
                    random_state=RANDOM_STATE
                )

                X_pca = pca.fit_transform(
                    X_train_scaled[:, variance_mask]
                )

                pca_means = np.mean(X_pca, axis=0)
                pca_embeddings[:n_components] = pca_means

                explained_var = np.sum(pca.explained_variance_ratio_)
                print(f" PCA: {n_components} components, explained variance: {explained_var:.4f}")

            except Exception as e:
                print(f" ⚠️ PCA failed: {e}")
        else:
            print(" ⚠️ PCA skipped (insufficient components)")
    else:
        print(" ⚠️ PCA skipped (insufficient samples or variance)")

    # ---------- CONCATENATE SIGNATURE ----------
    signature_vector = np.concatenate([
        mean_features,
        std_features,
        np.array([attack_ratio], dtype=np.float32),
        pca_embeddings
    ])

    device_signatures.append(signature_vector)

    signature_metadata.append({
        "device_num": device_num,
        "device_name": device_name,
        "n_features": X_train.shape[1],
        "signature_dim": signature_vector.shape[0],
        "attack_ratio_train": attack_ratio
    })

    print(f" ✓ Signature vector: {signature_vector.shape[0]} dimensions")

    # ---------- Cleanup ----------
    del (
        X_train, X_val, X_test,
        y_train, y_val, y_test,
        X_train_scaled, X_val_scaled, X_test_scaled,
        train_data, val_data, test_data,
        scaler
    )
    gc.collect()

# ---------- SAVE ALL SIGNATURES ----------
device_signatures_matrix = np.vstack(device_signatures)

print("\n" + "=" * 60)
print(f"✅ Device signatures matrix: {device_signatures_matrix.shape}")
print("=" * 60)

signatures_df = pd.DataFrame(
    device_signatures_matrix,
    index=[
        DEVICE_NAME_MAP.get(d["device_num"], f"Device_{d['device_num']}")
        for d in signature_metadata
    ]
)

signatures_df.insert(
    0,
    "device_num",
    [d["device_num"] for d in signature_metadata]
)

signatures_path = os.path.join(OUT_RESULTS, "device_signatures_train_only.csv")
signatures_df.to_csv(signatures_path, index=True)

metadata_df = pd.DataFrame(signature_metadata)
metadata_path = os.path.join(OUT_RESULTS, "signature_metadata.csv")
metadata_df.to_csv(metadata_path, index=False)

print(f"\n✓ Saved signatures to: {signatures_path}")
print(f"✓ Saved metadata to: {metadata_path}")

print("\n" + "=" * 60)
print("✅ SECTION 3 COMPLETE - NO DATA LEAKAGE")
print("=" * 60)

print("\nSignature metadata:")
print(metadata_df.to_string(index=False))
print("\n" + "=" * 60)


In [ ]:
# ============================================================
# SECTION 4: K-Means Clustering on Device Signatures
# ============================================================

print("\n" + "=" * 60)
print("SECTION 4: Device Clustering")
print("=" * 60)

# ---------- Load signatures ----------
signatures_path = os.path.join(OUT_RESULTS, "device_signatures_train_only.csv")
signatures_df = pd.read_csv(signatures_path, index_col=0)

# Extract device numbers and names
device_nums_from_sig = signatures_df['device_num'].values
device_names_from_sig = signatures_df.index.values

# Drop device_num column to get pure signature matrix
signatures_matrix = signatures_df.drop(columns=['device_num']).values

print(f"\n✓ Loaded signatures: {signatures_matrix.shape}")
print(f"  Devices: {list(device_nums_from_sig)}")

# ---------- Standardize signatures for clustering ----------
# Important: standardize signatures so all features contribute equally to distance
print("\n📊 Standardizing signatures for clustering...")

sig_scaler = StandardScaler()
signatures_scaled = sig_scaler.fit_transform(signatures_matrix)

print(f"  Scaled signature range: [{signatures_scaled.min():.4f}, {signatures_scaled.max():.4f}]")

# ---------- Determine optimal K using silhouette score ----------
print("\n🔍 Finding optimal number of clusters (K)...")

K_RANGE = range(2, min(8, len(device_nums_from_sig)))  # Try K from 2 to 7 (or fewer if less devices)
silhouette_scores = []

for k in K_RANGE:
    kmeans_temp = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=20)
    labels_temp = kmeans_temp.fit_predict(signatures_scaled)
    
    # Silhouette score measures cluster quality (-1 to 1, higher is better)
    sil_score = silhouette_score(signatures_scaled, labels_temp)
    silhouette_scores.append(sil_score)
    
    print(f"  K={k}: silhouette={sil_score:.4f}")

# Select K with highest silhouette score
best_k_idx = np.argmax(silhouette_scores)
best_k = list(K_RANGE)[best_k_idx]
best_silhouette = silhouette_scores[best_k_idx]

print(f"\n✓ Optimal K = {best_k} (silhouette = {best_silhouette:.4f})")

# ---------- Final clustering with optimal K ----------
print(f"\n🎯 Performing final clustering with K={best_k}...")

kmeans_final = KMeans(n_clusters=best_k, random_state=RANDOM_STATE, n_init=20)
cluster_labels = kmeans_final.fit_predict(signatures_scaled)

# ---------- Create cluster assignment DataFrame ----------
cluster_assignments = pd.DataFrame({
    'device_num': device_nums_from_sig,
    'device_name': device_names_from_sig,
    'cluster': cluster_labels
})

# Sort by cluster, then device_num
cluster_assignments = cluster_assignments.sort_values(['cluster', 'device_num'])

# ---------- Display cluster composition ----------
print("\n" + "=" * 60)
print("CLUSTER ASSIGNMENTS")
print("=" * 60)
print(cluster_assignments.to_string(index=False))

print("\n" + "=" * 60)
print("CLUSTER SUMMARY")
print("=" * 60)

for cluster_id in sorted(cluster_assignments['cluster'].unique()):
    cluster_devices = cluster_assignments[cluster_assignments['cluster'] == cluster_id]
    print(f"\n🔹 Cluster {cluster_id} ({len(cluster_devices)} devices):")
    for _, row in cluster_devices.iterrows():
        # Get attack ratio for this device from metadata
        attack_ratio = signature_metadata[int(row['device_num'])-1]['attack_ratio_train']
        print(f"  - Device {row['device_num']}: {row['device_name']:<45} (attack ratio: {attack_ratio:.3f})")

# ---------- Save results ----------
clusters_path = os.path.join(OUT_RESULTS, "device_clusters.csv")
cluster_assignments.to_csv(clusters_path, index=False)

silhouette_results = pd.DataFrame({
    'K': list(K_RANGE),
    'silhouette_score': silhouette_scores
})
silhouette_path = os.path.join(OUT_RESULTS, "clustering_silhouette_scores.csv")
silhouette_results.to_csv(silhouette_path, index=False)

# Save scaler and kmeans model for reproducibility
import joblib
joblib.dump(sig_scaler, os.path.join(OUT_RESULTS, "signature_scaler.joblib"))
joblib.dump(kmeans_final, os.path.join(OUT_RESULTS, "kmeans_model.joblib"))

print("\n" + "=" * 60)
print("✅ SECTION 4 COMPLETE")
print("=" * 60)
print(f"\n✓ Saved cluster assignments to: {clusters_path}")
print(f"✓ Saved silhouette scores to: {silhouette_path}")
print(f"✓ Saved clustering models to: {OUT_RESULTS}")
print(f"\nBest K: {best_k}")
print(f"Silhouette score: {best_silhouette:.4f}")
print("=" * 60)

In [ ]:
# ============================================================
# SECTION 4D: Smart Clustering (Prevent Singleton Clusters)
# ============================================================

import json
from collections import Counter

print("\n" + "=" * 60)
print("SECTION 4D: Smart Clustering Strategy")
print("=" * 60)

print("\n⚠️  Problem: K-Means creates singleton clusters")
print("Solution: Try multiple K values and choose based on cluster balance\n")

# Define what makes a "good" cluster configuration
def evaluate_clustering_quality(labels, n_devices=9):
    """
    Evaluate clustering based on:
    1. Minimum cluster size (avoid singletons)
    2. Balance (avoid one huge cluster)
    3. Silhouette score (secondary consideration)
    """
    cluster_sizes = Counter(labels)
    
    min_size = min(cluster_sizes.values())
    max_size = max(cluster_sizes.values())
    n_singletons = sum(1 for size in cluster_sizes.values() if size == 1)
    balance_ratio = max_size / n_devices  # Lower is better
    
    return {
        'min_size': min_size,
        'max_size': max_size,
        'n_singletons': n_singletons,
        'balance_ratio': balance_ratio,
        'sizes': {int(k): int(v) for k, v in cluster_sizes.items()}  # <-- CONVERT TO PYTHON INT
    }

# Test K=2 and K=3 with quality metrics
print("Evaluating clustering options:\n")

best_config = None
best_score = -999

for test_k in [2, 3]:
    print(f"{'─' * 60}")
    print(f"K = {test_k}")
    print(f"{'─' * 60}")
    
    kmeans_test = KMeans(n_clusters=test_k, random_state=RANDOM_STATE, n_init=20)
    labels = kmeans_test.fit_predict(signatures_scaled)
    sil = silhouette_score(signatures_scaled, labels)
    quality = evaluate_clustering_quality(labels)
    
    print(f"Silhouette: {sil:.4f}")
    print(f"Cluster sizes: {quality['sizes']}")
    print(f"Singleton clusters: {quality['n_singletons']}")
    print(f"Min cluster size: {quality['min_size']}")
    print(f"Balance ratio: {quality['balance_ratio']:.2f}")
    
    # Scoring: penalize singletons heavily, prefer balance
    score = (
        sil * 10 +  # Silhouette (0.1 = 1 point)
        (quality['min_size'] >= 2) * 50 -  # Huge bonus for no singletons
        quality['n_singletons'] * 30 -  # Heavy penalty for singletons
        quality['balance_ratio'] * 20  # Penalty for imbalance
    )
    
    print(f"Quality score: {score:.2f}")
    
    if score > best_score:
        best_score = score
        best_config = {
            'k': test_k,
            'labels': labels,
            'silhouette': sil,
            'quality': quality,
            'kmeans': kmeans_test
        }
    print()

print("=" * 60)
print(f"🎯 BEST CHOICE: K = {best_config['k']}")
print("=" * 60)
print(f"Silhouette: {best_config['silhouette']:.4f}")
print(f"Cluster sizes: {best_config['quality']['sizes']}")
print(f"Singleton clusters: {best_config['quality']['n_singletons']}")
print(f"Quality score: {best_score:.2f}")

# Use the best configuration
FINAL_K = best_config['k']
cluster_labels = best_config['labels']
kmeans_final = best_config['kmeans']

# Create final assignments
cluster_assignments = pd.DataFrame({
    'device_num': device_nums_from_sig,
    'device_name': device_names_from_sig,
    'cluster': cluster_labels
})
cluster_assignments = cluster_assignments.sort_values(['cluster', 'device_num'])

print("\n" + "=" * 60)
print(f"FINAL CLUSTER ASSIGNMENTS (K={FINAL_K})")
print("=" * 60)
print(cluster_assignments.to_string(index=False))

print("\n" + "=" * 60)
print("CLUSTER COMPOSITION")
print("=" * 60)

for cluster_id in sorted(cluster_assignments['cluster'].unique()):
    cluster_devices = cluster_assignments[cluster_assignments['cluster'] == cluster_id]
    print(f"\n🔹 Cluster {cluster_id} ({len(cluster_devices)} device(s)):")
    for _, row in cluster_devices.iterrows():
        attack_ratio = signature_metadata[int(row['device_num'])-1]['attack_ratio_train']
        print(f"  - Device {row['device_num']}: {row['device_name']:<45} (attack: {attack_ratio:.3f})")

# Save everything
clusters_path = os.path.join(OUT_RESULTS, "device_clusters_final.csv")
cluster_assignments.to_csv(clusters_path, index=False)

joblib.dump(sig_scaler, os.path.join(OUT_RESULTS, "signature_scaler.joblib"))
joblib.dump(kmeans_final, os.path.join(OUT_RESULTS, "kmeans_model.joblib"))

clustering_summary = {
    'chosen_k': int(FINAL_K),
    'silhouette_score': float(best_config['silhouette']),
    'cluster_sizes': best_config['quality']['sizes'],
    'n_singletons': int(best_config['quality']['n_singletons']),
    'selection_criteria': 'Prioritized cluster balance over silhouette score'
}

with open(os.path.join(OUT_RESULTS, "clustering_summary.json"), 'w') as f:
    json.dump(clustering_summary, f, indent=2)

print("\n" + "=" * 60)
print("✅ SECTION 4 COMPLETE (SMART K SELECTION)")
print("=" * 60)
print(f"✓ Selected K={FINAL_K} based on cluster balance")
print(f"✓ Silhouette: {best_config['silhouette']:.4f}")
print(f"✓ All clusters have size >= {best_config['quality']['min_size']}")
print(f"✓ Ready for two-tier federated learning!")
print("=" * 60)

In [ ]:
# ============================================================
# SECTION 5: Per-Device Local Model Training
# ============================================================

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import warnings
warnings.filterwarnings('ignore')

# Reduce TensorFlow logging
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
tf.get_logger().setLevel('ERROR')

print("\n" + "=" * 60)
print("SECTION 5: Per-Device Local Model Training")
print("=" * 60)

# ---------- Model Architecture (MLP) ----------
def build_mlp(input_dim, name="mlp"):
    """
    Build MLP architecture matching the methodology paper
    Architecture: Input → Dense(128) → Dense(64) → Dense(1)
    """
    model = models.Sequential([
        layers.Input(shape=(input_dim,), name='input'),
        layers.Dense(128, activation='relu', name='hidden1'),
        layers.Dense(64, activation='relu', name='hidden2'),
        layers.Dense(1, activation='sigmoid', name='output')
    ], name=name)
    
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    
    return model

# ---------- Training Configuration ----------
EPOCHS = 10
BATCH_SIZE = 1024
VERBOSE = 0  # Silent training (set to 1 to see progress bars)

# ---------- Storage for results ----------
device_training_results = []
OUT_MODELS = os.path.join(OUT_RESULTS, "local_models")
os.makedirs(OUT_MODELS, exist_ok=True)

# ---------- Load cluster assignments ----------
clusters_df = pd.read_csv(os.path.join(OUT_RESULTS, "device_clusters_final.csv"))

# ---------- Train each device ----------
print("\n🚀 Training local models on each device...\n")

for device_num in device_nums:
    device_name = DEVICE_NAME_MAP.get(device_num, f"Unknown_{device_num}")
    device_dir = os.path.join(OUT_CLIENTS, f"device_{device_num}_{device_name}")
    
    print(f"{'─' * 60}")
    print(f"📱 Device {device_num}: {device_name}")
    print(f"{'─' * 60}")
    
    # Get cluster assignment
    cluster_id = clusters_df[clusters_df['device_num'] == int(device_num)]['cluster'].values[0]
    print(f"  Cluster: {cluster_id}")
    
    # ---------- Load scaled training and validation data ----------
    train_data = np.load(os.path.join(device_dir, "train_scaled.npz"))
    val_data = np.load(os.path.join(device_dir, "val_scaled.npz"))
    
    X_train = train_data['X']
    y_train = train_data['y']
    X_val = val_data['X']
    y_val = val_data['y']
    
    n_train = X_train.shape[0]
    n_val = X_val.shape[0]
    input_dim = X_train.shape[1]
    
    print(f"  Training samples: {n_train:,}")
    print(f"  Validation samples: {n_val:,}")
    print(f"  Input features: {input_dim}")
    
    # ---------- Calculate class weights (CORRECTED FORMULA) ----------
    n_pos = int(y_train.sum())
    n_neg = n_train - n_pos
    
    if n_pos > 0 and n_neg > 0:
        # Correct formula: total / (2 * class_count)
        total = n_pos + n_neg
        class_weight = {
            0: total / (2.0 * n_neg),  # Benign (minority in most cases)
            1: total / (2.0 * n_pos)   # Malicious (majority in most cases)
        }
        print(f"  Class distribution: {n_neg} benign, {n_pos} malicious")
        print(f"  Class weights: benign={class_weight[0]:.3f}, malicious={class_weight[1]:.3f}")
    else:
        class_weight = None
        print(f"  ⚠️  Single-class data, skipping class weighting")
    
    # ---------- Build and train model ----------
    model = build_mlp(input_dim, name=f"device_{device_num}_model")
    
    print(f"\n  Training for {EPOCHS} epochs...")
    
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val) if n_val > 0 else None,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        class_weight=class_weight,
        verbose=VERBOSE
    )
    
    # ---------- Evaluate on validation set ----------
    if n_val > 0:
        y_val_pred_prob = model.predict(X_val, verbose=0)
        y_val_pred = (y_val_pred_prob >= 0.5).astype(int).flatten()
        
        val_acc = accuracy_score(y_val, y_val_pred)
        val_prec = precision_score(y_val, y_val_pred, zero_division=0)
        val_rec = recall_score(y_val, y_val_pred, zero_division=0)
        val_f1 = f1_score(y_val, y_val_pred, zero_division=0)
        
        print(f"\n  ✓ Validation Results:")
        print(f"    Accuracy:  {val_acc:.4f}")
        print(f"    Precision: {val_prec:.4f}")
        print(f"    Recall:    {val_rec:.4f}")
        print(f"    F1-Score:  {val_f1:.4f}")
    else:
        val_acc = val_prec = val_rec = val_f1 = None
        print(f"\n  ⚠️  No validation data available")
    
    # ---------- Save model weights ----------
    weights_path = os.path.join(OUT_MODELS, f"device_{device_num}_{device_name}.weights.h5")
    model.save_weights(weights_path)
    print(f"\n  ✅ Saved weights: {weights_path}")
    
    # ---------- Store results ----------
    device_training_results.append({
        'device_num': device_num,
        'device_name': device_name,
        'cluster': int(cluster_id),
        'n_train': n_train,
        'n_val': n_val,
        'n_train_pos': n_pos,
        'n_train_neg': n_neg,
        'val_accuracy': val_acc,
        'val_precision': val_prec,
        'val_recall': val_rec,
        'val_f1': val_f1,
        'final_train_loss': float(history.history['loss'][-1]),
        'final_train_acc': float(history.history['accuracy'][-1])
    })
    
    # Cleanup
    del model, X_train, y_train, X_val, y_val, train_data, val_data
    keras.backend.clear_session()
    gc.collect()
    
    print()

# ---------- Save training summary ----------
results_df = pd.DataFrame(device_training_results)
results_path = os.path.join(OUT_RESULTS, "local_model_training_results.csv")
results_df.to_csv(results_path, index=False)

print("=" * 60)
print("✅ SECTION 5 COMPLETE")
print("=" * 60)
print(f"\n✓ Trained {len(device_nums)} local models")
print(f"✓ Saved weights to: {OUT_MODELS}")
print(f"✓ Training results: {results_path}")

print("\n📊 Training Summary:")
print(results_df[['device_num', 'device_name', 'cluster', 'val_accuracy', 'val_f1']].to_string(index=False))

print("\n🎯 Next Steps:")
print("  - Section 6: Cluster-level federated aggregation (Tier 1)")
print("  - Section 7: Global federated aggregation (Tier 2)")
print("=" * 60)

In [ ]:
# ============================================================
# SECTION 6: Cluster-Level Federated Aggregation (Tier 1)
# ============================================================

print("\n" + "=" * 60)
print("SECTION 6: Cluster-Level Federated Aggregation (Tier 1)")
print("=" * 60)

# ---------- FedAvg Implementation ----------
def federated_averaging(weights_list):
    """
    Perform Federated Averaging (FedAvg) on a list of model weights.
    
    Args:
        weights_list: List of weight arrays from different models
    
    Returns:
        Averaged weights
    """
    if len(weights_list) == 0:
        raise ValueError("Cannot average empty weights list")
    
    if len(weights_list) == 1:
        print("    ℹ️  Only 1 model in cluster, using its weights directly")
        return weights_list[0]
    
    # Average across all models
    avg_weights = []
    for layer_weights in zip(*weights_list):
        # Average weights for this layer
        avg_weights.append(np.mean(layer_weights, axis=0))
    
    return avg_weights

# ---------- Load cluster assignments ----------
clusters_df = pd.read_csv(os.path.join(OUT_RESULTS, "device_clusters_final.csv"))

# Get unique clusters
unique_clusters = sorted(clusters_df['cluster'].unique())

print(f"\n📊 Found {len(unique_clusters)} clusters")
print(f"   Clusters: {list(unique_clusters)}")

# Determine input dimension from any device
sample_device = device_nums[0]
sample_device_name = DEVICE_NAME_MAP[sample_device]
sample_dir = os.path.join(OUT_CLIENTS, f"device_{sample_device}_{sample_device_name}")
sample_data = np.load(os.path.join(sample_dir, "train_scaled.npz"))
INPUT_DIM = sample_data['X'].shape[1]
print(f"   Input dimension: {INPUT_DIM}")

# Create output directory for cluster models
OUT_CLUSTER_MODELS = os.path.join(OUT_RESULTS, "cluster_models")
os.makedirs(OUT_CLUSTER_MODELS, exist_ok=True)

# ---------- Aggregate each cluster ----------
print("\n🔄 Performing cluster-level aggregation...\n")

cluster_model_paths = {}

for cluster_id in unique_clusters:
    print(f"{'─' * 60}")
    print(f"🔹 Cluster {cluster_id}")
    print(f"{'─' * 60}")
    
    # Get devices in this cluster
    cluster_devices = clusters_df[clusters_df['cluster'] == cluster_id]
    device_nums_in_cluster = cluster_devices['device_num'].values
    
    print(f"  Devices in cluster: {len(device_nums_in_cluster)}")
    for dev_num in device_nums_in_cluster:
        dev_name = DEVICE_NAME_MAP.get(str(dev_num), f"Unknown_{dev_num}")
        print(f"    - Device {dev_num}: {dev_name}")
    
    # Load weights from all devices in this cluster
    print(f"\n  Loading device model weights...")
    weights_list = []
    
    for dev_num in device_nums_in_cluster:
        dev_name = DEVICE_NAME_MAP.get(str(dev_num), f"Unknown_{dev_num}")
        weights_path = os.path.join(OUT_MODELS, f"device_{dev_num}_{dev_name}.weights.h5")
        
        # Create a temporary model to load weights
        temp_model = build_mlp(INPUT_DIM, name=f"temp_device_{dev_num}")
        temp_model.load_weights(weights_path)
        
        # Extract weights
        device_weights = temp_model.get_weights()
        weights_list.append(device_weights)
        
        print(f"    ✓ Loaded weights from device {dev_num}")
        
        # Cleanup
        del temp_model
        keras.backend.clear_session()
    
    # Perform FedAvg
    print(f"\n  Aggregating {len(weights_list)} model(s)...")
    aggregated_weights = federated_averaging(weights_list)
    
    # Create cluster model and set aggregated weights
    cluster_model = build_mlp(INPUT_DIM, name=f"cluster_{cluster_id}_model")
    cluster_model.set_weights(aggregated_weights)
    
    # Save cluster model
    cluster_weights_path = os.path.join(OUT_CLUSTER_MODELS, f"cluster_{cluster_id}.weights.h5")
    cluster_model.save_weights(cluster_weights_path)
    cluster_model_paths[cluster_id] = cluster_weights_path
    
    print(f"  ✅ Saved cluster model: {cluster_weights_path}")
    
    # Cleanup
    del cluster_model, weights_list, aggregated_weights
    keras.backend.clear_session()
    gc.collect()
    
    print()

# ---------- Save cluster model summary ----------
cluster_summary = {
    'n_clusters': len(unique_clusters),
    'cluster_details': []
}

for cluster_id in unique_clusters:
    cluster_devices = clusters_df[clusters_df['cluster'] == cluster_id]
    cluster_summary['cluster_details'].append({
        'cluster_id': int(cluster_id),
        'n_devices': int(len(cluster_devices)),
        'devices': [int(x) for x in cluster_devices['device_num'].values],
        'weights_path': cluster_model_paths[cluster_id]
    })

with open(os.path.join(OUT_CLUSTER_MODELS, "cluster_aggregation_summary.json"), 'w') as f:
    json.dump(cluster_summary, f, indent=2)

print("=" * 60)
print("✅ SECTION 6 COMPLETE (TIER 1 FL)")
print("=" * 60)
print(f"\n✓ Aggregated {len(unique_clusters)} cluster models")
print(f"✓ Cluster models saved to: {OUT_CLUSTER_MODELS}")
print("\n📊 Cluster Summary:")
for detail in cluster_summary['cluster_details']:
    print(f"  Cluster {detail['cluster_id']}: {detail['n_devices']} device(s) → {len(detail['devices'])} model(s) averaged")

print("\n🎯 Next Step:")
print("  - Section 7: Global federated aggregation (Tier 2 FL)")
print("=" * 60)

In [ ]:
# ============================================================
# SECTION 7: Global Federated Aggregation (Tier 2 FL)
# ============================================================

print("\n" + "=" * 60)
print("SECTION 7: Global Federated Aggregation (Tier 2)")
print("=" * 60)

print("\n🌍 Creating global model by aggregating cluster models...")

# Create output directory for global model
OUT_GLOBAL_MODEL = os.path.join(OUT_RESULTS, "global_model")
os.makedirs(OUT_GLOBAL_MODEL, exist_ok=True)

# ---------- Load cluster model summary ----------
with open(os.path.join(OUT_CLUSTER_MODELS, "cluster_aggregation_summary.json"), 'r') as f:
    cluster_summary = json.load(f)

n_clusters = cluster_summary['n_clusters']
print(f"\n📊 Loading {n_clusters} cluster models...")

# ---------- Load cluster model weights ----------
cluster_weights_list = []

for cluster_detail in cluster_summary['cluster_details']:
    cluster_id = cluster_detail['cluster_id']
    weights_path = cluster_detail['weights_path']
    n_devices = cluster_detail['n_devices']
    
    print(f"\n  🔹 Cluster {cluster_id}:")
    print(f"     Devices: {n_devices}")
    print(f"     Weights: {weights_path}")
    
    # Load cluster model
    cluster_model = build_mlp(INPUT_DIM, name=f"cluster_{cluster_id}")
    cluster_model.load_weights(weights_path)
    
    # Extract weights
    cluster_weights = cluster_model.get_weights()
    cluster_weights_list.append(cluster_weights)
    
    print(f"     ✓ Loaded")
    
    # Cleanup
    del cluster_model
    keras.backend.clear_session()

# ---------- Perform Global FedAvg ----------
print(f"\n🔄 Aggregating {len(cluster_weights_list)} cluster models using FedAvg...")

global_weights = federated_averaging(cluster_weights_list)

# ---------- Create and save global model ----------
global_model = build_mlp(INPUT_DIM, name="global_intrusion_detection_model")
global_model.set_weights(global_weights)

global_weights_path = os.path.join(OUT_GLOBAL_MODEL, "global_model.weights.h5")
global_model.save_weights(global_weights_path)

print(f"\n✅ Global model saved: {global_weights_path}")

# ---------- Save global model summary ----------
global_summary = {
    'model_type': 'hierarchical_federated_learning',
    'architecture': {
        'type': 'MLP',
        'layers': [
            {'name': 'input', 'units': INPUT_DIM},
            {'name': 'hidden1', 'units': 128, 'activation': 'relu'},
            {'name': 'hidden2', 'units': 64, 'activation': 'relu'},
            {'name': 'output', 'units': 1, 'activation': 'sigmoid'}
        ]
    },
    'training_details': {
        'total_devices': 9,
        'n_clusters': n_clusters,
        'tier1_aggregation': 'cluster_level_fedavg',
        'tier2_aggregation': 'global_fedavg',
        'local_epochs': EPOCHS,
        'batch_size': BATCH_SIZE
    },
    'aggregation_hierarchy': {
        'tier1': [
            {
                'cluster_id': detail['cluster_id'],
                'n_devices': detail['n_devices'],
                'devices': detail['devices']
            }
            for detail in cluster_summary['cluster_details']
        ],
        'tier2': {
            'n_clusters': n_clusters,
            'global_model_path': global_weights_path
        }
    }
}

with open(os.path.join(OUT_GLOBAL_MODEL, "global_model_summary.json"), 'w') as f:
    json.dump(global_summary, f, indent=2)

# Cleanup
del global_model, cluster_weights_list, global_weights
keras.backend.clear_session()
gc.collect()

print("\n" + "=" * 60)
print("✅ SECTION 7 COMPLETE (TIER 2 FL)")
print("=" * 60)

print("\n🎉 TWO-TIER HIERARCHICAL FEDERATED LEARNING COMPLETE!")
print("\n📊 Model Hierarchy:")
print("   ┌─────────────────────────────────────┐")
print("   │      GLOBAL MODEL (Tier 2)          │")
print("   │   (Average of 2 cluster models)     │")
print("   └─────────────────────────────────────┘")
print("              ▲          ▲")
print("              │          │")
print("      ┌───────┴───┐  ┌──┴──────────────┐")
print("      │ Cluster 0 │  │   Cluster 1     │")
print("      │ (1 device)│  │  (8 devices)    │")
print("      └───────────┘  └─────────────────┘")
print("           │              │")
print("       Device 4      Devices 1,2,3,5,6,7,8,9")

print("\n✅ Models Created:")
print(f"   - Local models: 9 (saved in {OUT_MODELS})")
print(f"   - Cluster models: 2 (saved in {OUT_CLUSTER_MODELS})")
print(f"   - Global model: 1 (saved in {OUT_GLOBAL_MODEL})")

print("\n🎯 Next Steps:")
print("   - Section 8: Comprehensive evaluation on test sets")
print("   - Compare: Local vs Cluster vs Global models")
print("   - Baseline: Centralized training vs Hierarchical FL")
print("=" * 60)

In [ ]:
# ============================================================
# SECTION 8: Comprehensive Evaluation on Test Sets
# ============================================================

from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

print("\n" + "=" * 60)
print("SECTION 8: Comprehensive Model Evaluation")
print("=" * 60)
print("\n⚠️  IMPORTANT: Using TEST sets (held-out data, never seen during training)")

# ---------- Storage for evaluation results ----------
evaluation_results = []

# ---------- Load cluster assignments ----------
clusters_df = pd.read_csv(os.path.join(OUT_RESULTS, "device_clusters_final.csv"))

# ---------- Evaluate each model type on each device's test set ----------
print("\n🔍 Evaluating models on test data...\n")

for device_num in device_nums:
    device_name = DEVICE_NAME_MAP.get(device_num, f"Unknown_{device_num}")
    device_dir = os.path.join(OUT_CLIENTS, f"device_{device_num}_{device_name}")
    cluster_id = int(clusters_df[clusters_df['device_num'] == int(device_num)]['cluster'].values[0])
    
    print(f"{'─' * 60}")
    print(f"📱 Device {device_num}: {device_name} (Cluster {cluster_id})")
    print(f"{'─' * 60}")
    
    # ---------- Load test data ----------
    test_data = np.load(os.path.join(device_dir, "test_scaled.npz"))
    X_test = test_data['X']
    y_test = test_data['y']
    
    n_test = len(y_test)
    n_test_pos = int(y_test.sum())
    n_test_neg = n_test - n_test_pos
    
    print(f"  Test samples: {n_test:,} (benign: {n_test_neg}, malicious: {n_test_pos})")
    
    if n_test == 0:
        print(f"  ⚠️  No test data, skipping evaluation")
        continue
    
    # ---------- 1. Evaluate LOCAL model ----------
    print(f"\n  📊 Evaluating LOCAL model (Device {device_num})...")
    
    local_model = build_mlp(INPUT_DIM, name=f"local_{device_num}")
    local_model.load_weights(os.path.join(OUT_MODELS, f"device_{device_num}_{device_name}.weights.h5"))
    
    y_pred_local_prob = local_model.predict(X_test, verbose=0)
    y_pred_local = (y_pred_local_prob >= 0.5).astype(int).flatten()
    
    local_acc = accuracy_score(y_test, y_pred_local)
    local_prec = precision_score(y_test, y_pred_local, zero_division=0)
    local_rec = recall_score(y_test, y_pred_local, zero_division=0)
    local_f1 = f1_score(y_test, y_pred_local, zero_division=0)
    
    print(f"     Accuracy: {local_acc:.4f}, Precision: {local_prec:.4f}, Recall: {local_rec:.4f}, F1: {local_f1:.4f}")
    
    del local_model
    keras.backend.clear_session()
    
    # ---------- 2. Evaluate CLUSTER model ----------
    print(f"\n  📊 Evaluating CLUSTER model (Cluster {cluster_id})...")
    
    cluster_model = build_mlp(INPUT_DIM, name=f"cluster_{cluster_id}")
    cluster_model.load_weights(os.path.join(OUT_CLUSTER_MODELS, f"cluster_{cluster_id}.weights.h5"))
    
    y_pred_cluster_prob = cluster_model.predict(X_test, verbose=0)
    y_pred_cluster = (y_pred_cluster_prob >= 0.5).astype(int).flatten()
    
    cluster_acc = accuracy_score(y_test, y_pred_cluster)
    cluster_prec = precision_score(y_test, y_pred_cluster, zero_division=0)
    cluster_rec = recall_score(y_test, y_pred_cluster, zero_division=0)
    cluster_f1 = f1_score(y_test, y_pred_cluster, zero_division=0)
    
    print(f"     Accuracy: {cluster_acc:.4f}, Precision: {cluster_prec:.4f}, Recall: {cluster_rec:.4f}, F1: {cluster_f1:.4f}")
    
    del cluster_model
    keras.backend.clear_session()
    
    # ---------- 3. Evaluate GLOBAL model ----------
    print(f"\n  📊 Evaluating GLOBAL model...")
    
    global_model = build_mlp(INPUT_DIM, name="global")
    global_model.load_weights(os.path.join(OUT_GLOBAL_MODEL, "global_model.weights.h5"))
    
    y_pred_global_prob = global_model.predict(X_test, verbose=0)
    y_pred_global = (y_pred_global_prob >= 0.5).astype(int).flatten()
    
    global_acc = accuracy_score(y_test, y_pred_global)
    global_prec = precision_score(y_test, y_pred_global, zero_division=0)
    global_rec = recall_score(y_test, y_pred_global, zero_division=0)
    global_f1 = f1_score(y_test, y_pred_global, zero_division=0)
    
    print(f"     Accuracy: {global_acc:.4f}, Precision: {global_prec:.4f}, Recall: {global_rec:.4f}, F1: {global_f1:.4f}")
    
    del global_model
    keras.backend.clear_session()
    
    # ---------- Store results ----------
    evaluation_results.append({
        'device_num': device_num,
        'device_name': device_name,
        'cluster': cluster_id,
        'n_test': n_test,
        'n_test_benign': n_test_neg,
        'n_test_malicious': n_test_pos,
        # Local model
        'local_accuracy': local_acc,
        'local_precision': local_prec,
        'local_recall': local_rec,
        'local_f1': local_f1,
        # Cluster model
        'cluster_accuracy': cluster_acc,
        'cluster_precision': cluster_prec,
        'cluster_recall': cluster_rec,
        'cluster_f1': cluster_f1,
        # Global model
        'global_accuracy': global_acc,
        'global_precision': global_prec,
        'global_recall': global_rec,
        'global_f1': global_f1
    })
    
    # Cleanup
    del X_test, y_test, test_data
    gc.collect()
    print()

# ---------- Save evaluation results ----------
eval_df = pd.DataFrame(evaluation_results)
eval_path = os.path.join(OUT_RESULTS, "comprehensive_evaluation_results.csv")
eval_df.to_csv(eval_path, index=False)

print("=" * 60)
print("✅ EVALUATION COMPLETE")
print("=" * 60)

# ---------- Display results ----------
print("\n📊 COMPREHENSIVE EVALUATION RESULTS")
print("=" * 60)

print("\n🎯 Per-Device Test Performance:\n")
display_cols = ['device_num', 'device_name', 'cluster', 
                'local_f1', 'cluster_f1', 'global_f1']
print(eval_df[display_cols].to_string(index=False))

# ---------- Aggregate statistics ----------
print("\n" + "=" * 60)
print("📈 AGGREGATE STATISTICS (Across All Devices)")
print("=" * 60)

print("\n🏆 Mean Performance:")
print(f"  Local Model  - Accuracy: {eval_df['local_accuracy'].mean():.4f}, F1: {eval_df['local_f1'].mean():.4f}")
print(f"  Cluster Model- Accuracy: {eval_df['cluster_accuracy'].mean():.4f}, F1: {eval_df['cluster_f1'].mean():.4f}")
print(f"  Global Model - Accuracy: {eval_df['global_accuracy'].mean():.4f}, F1: {eval_df['global_f1'].mean():.4f}")

print("\n📊 Standard Deviation:")
print(f"  Local Model  - Accuracy: {eval_df['local_accuracy'].std():.4f}, F1: {eval_df['local_f1'].std():.4f}")
print(f"  Cluster Model- Accuracy: {eval_df['cluster_accuracy'].std():.4f}, F1: {eval_df['cluster_f1'].std():.4f}")
print(f"  Global Model - Accuracy: {eval_df['global_accuracy'].std():.4f}, F1: {eval_df['global_f1'].std():.4f}")

print("\n✅ Results saved to:", eval_path)
print("=" * 60)

In [ ]:
# ============================================================
# COMPREHENSIVE EVALUATION & DATA COLLECTION (FIXED)
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, auc
)
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
import json
import os
import gc
import warnings
warnings.filterwarnings('ignore')

print("\n" + "=" * 60)
print("COMPREHENSIVE EVALUATION - DATA COLLECTION")
print("=" * 60)

# ---------- RE-DECLARE PATHS AND VARIABLES ----------
OUT_RESULTS = "/kaggle/working/results"
OUT_CLIENTS = "/kaggle/working/fl_clients"
OUT_MODELS = os.path.join(OUT_RESULTS, "local_models")
OUT_CLUSTER_MODELS = os.path.join(OUT_RESULTS, "cluster_models")
OUT_GLOBAL_MODEL = os.path.join(OUT_RESULTS, "global_model")

DEVICE_NAME_MAP = {
    '1': 'Danmini_Doorbell',
    '2': 'Ecobee_Thermostat',
    '3': 'Ennio_Doorbell',
    '4': 'Philips_B120N10_Baby_Monitor',
    '5': 'Provision_PT_737E_Security_Camera',
    '6': 'Provision_PT_838_Security_Camera',
    '7': 'Samsung_SNH_1011_N_Webcam',
    '8': 'SimpleHome_XCS7_1002_WHT_Security_Camera',
    '9': 'SimpleHome_XCS7_1003_WHT_Security_Camera'
}

device_nums = ['1', '2', '3', '4', '5', '6', '7', '8', '9']
INPUT_DIM = 115

# ---------- Rebuild MLP Function ----------
def build_mlp(input_dim, name="mlp"):
    """Build MLP architecture"""
    model = models.Sequential([
        layers.Input(shape=(input_dim,), name='input'),
        layers.Dense(128, activation='relu', name='hidden1'),
        layers.Dense(64, activation='relu', name='hidden2'),
        layers.Dense(1, activation='sigmoid', name='output')
    ], name=name)
    
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    
    return model

# ---------- Helper Functions ----------
def evaluate_model(model, X, y, model_name="Model"):
    """Comprehensive evaluation of a model"""
    y_pred_prob = model.predict(X, verbose=0)
    y_pred = (y_pred_prob >= 0.5).astype(int).flatten()
    
    acc = accuracy_score(y, y_pred)
    prec = precision_score(y, y_pred, zero_division=0)
    rec = recall_score(y, y_pred, zero_division=0)
    f1 = f1_score(y, y_pred, zero_division=0)
    
    # Confusion matrix
    cm = confusion_matrix(y, y_pred)
    tn, fp, fn, tp = cm.ravel() if cm.size == 4 else (0, 0, 0, 0)
    
    # False Positive Rate and True Positive Rate
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    tpr = tp / (tp + fn) if (tp + fn) > 0 else 0
    
    return {
        'model_name': model_name,
        'accuracy': acc,
        'precision': prec,
        'recall': rec,
        'f1_score': f1,
        'tp': int(tp),
        'tn': int(tn),
        'fp': int(fp),
        'fn': int(fn),
        'fpr': fpr,
        'tpr': tpr,
        'y_true': y,
        'y_pred': y_pred,
        'y_pred_prob': y_pred_prob.flatten(),
        'confusion_matrix': cm
    }

# ---------- Storage ----------
all_results = {
    'per_device_local': [],
    'per_device_cluster': [],
    'per_device_global': [],
    'roc_data': []
}

# Load cluster assignments
clusters_df = pd.read_csv(os.path.join(OUT_RESULTS, "device_clusters_final.csv"))

# ---------- Evaluate All Models on All Devices ----------
print("\n🔍 Evaluating all model types on all devices...\n")

for device_num in device_nums:
    device_name = DEVICE_NAME_MAP.get(device_num, f"Unknown_{device_num}")
    device_dir = os.path.join(OUT_CLIENTS, f"device_{device_num}_{device_name}")
    cluster_id = int(clusters_df[clusters_df['device_num'] == int(device_num)]['cluster'].values[0])
    
    print(f"{'─' * 60}")
    print(f"📱 Device {device_num}: {device_name} (Cluster {cluster_id})")
    
    # Load test data
    test_data = np.load(os.path.join(device_dir, "test_scaled.npz"))
    X_test = test_data['X']
    y_test = test_data['y']
    
    if len(y_test) == 0:
        print("  ⚠️ No test data, skipping")
        continue
    
    print(f"  Test samples: {len(y_test):,}")
    
    # 1. LOCAL MODEL
    print("  📊 Evaluating LOCAL model...")
    local_model = build_mlp(INPUT_DIM, name=f"local_{device_num}")
    local_model.load_weights(os.path.join(OUT_MODELS, f"device_{device_num}_{device_name}.weights.h5"))
    
    local_results = evaluate_model(local_model, X_test, y_test, f"Local_Device_{device_num}")
    local_results.update({
        'device_num': device_num,
        'device_name': device_name,
        'cluster': cluster_id,
        'n_test': len(y_test),
        'n_test_benign': int((y_test == 0).sum()),
        'n_test_malicious': int((y_test == 1).sum())
    })
    all_results['per_device_local'].append(local_results)
    
    del local_model
    keras.backend.clear_session()
    
    # 2. CLUSTER MODEL
    print("  📊 Evaluating CLUSTER model...")
    cluster_model = build_mlp(INPUT_DIM, name=f"cluster_{cluster_id}")
    cluster_model.load_weights(os.path.join(OUT_CLUSTER_MODELS, f"cluster_{cluster_id}.weights.h5"))
    
    cluster_results = evaluate_model(cluster_model, X_test, y_test, f"Cluster_{cluster_id}_Device_{device_num}")
    cluster_results.update({
        'device_num': device_num,
        'device_name': device_name,
        'cluster': cluster_id,
        'n_test': len(y_test)
    })
    all_results['per_device_cluster'].append(cluster_results)
    
    del cluster_model
    keras.backend.clear_session()
    
    # 3. GLOBAL MODEL
    print("  📊 Evaluating GLOBAL model...")
    global_model = build_mlp(INPUT_DIM, name="global")
    global_model.load_weights(os.path.join(OUT_GLOBAL_MODEL, "global_model.weights.h5"))
    
    global_results = evaluate_model(global_model, X_test, y_test, f"Global_Device_{device_num}")
    global_results.update({
        'device_num': device_num,
        'device_name': device_name,
        'cluster': cluster_id,
        'n_test': len(y_test)
    })
    all_results['per_device_global'].append(global_results)
    
    # Store ROC data for this device
    for model_type, results in [('Local', local_results), ('Cluster', cluster_results), ('Global', global_results)]:
        fpr_roc, tpr_roc, _ = roc_curve(results['y_true'], results['y_pred_prob'])
        roc_auc = auc(fpr_roc, tpr_roc)
        
        all_results['roc_data'].append({
            'device_num': device_num,
            'device_name': device_name,
            'model_type': model_type,
            'fpr': fpr_roc,
            'tpr': tpr_roc,
            'auc': roc_auc
        })
    
    del global_model, X_test, y_test, test_data
    keras.backend.clear_session()
    gc.collect()
    print()

# ---------- Save Raw Results ----------
with open(os.path.join(OUT_RESULTS, "evaluation_raw_data.json"), 'w') as f:
    # Convert numpy arrays to lists for JSON serialization
    save_data = {
        'per_device_local': [{k: (v.tolist() if isinstance(v, np.ndarray) else v) 
                              for k, v in item.items() if k not in ['y_true', 'y_pred', 'y_pred_prob', 'confusion_matrix']} 
                             for item in all_results['per_device_local']],
        'per_device_cluster': [{k: (v.tolist() if isinstance(v, np.ndarray) else v) 
                                for k, v in item.items() if k not in ['y_true', 'y_pred', 'y_pred_prob', 'confusion_matrix']} 
                               for item in all_results['per_device_cluster']],
        'per_device_global': [{k: (v.tolist() if isinstance(v, np.ndarray) else v) 
                               for k, v in item.items() if k not in ['y_true', 'y_pred', 'y_pred_prob', 'confusion_matrix']} 
                              for item in all_results['per_device_global']]
    }
    json.dump(save_data, f, indent=2)

print("=" * 60)
print("✅ DATA COLLECTION COMPLETE")
print("=" * 60)
print(f"\n✓ Evaluated {len(all_results['per_device_local'])} devices")
print(f"✓ Saved raw data to: {OUT_RESULTS}/evaluation_raw_data.json")
print("\n🎯 Ready to generate tables and figures!")
print("=" * 60)

In [ ]:
# ============================================================
# TABLE 1: Test Set Distribution per Device
# ============================================================

print("\n" + "=" * 60)
print("GENERATING TABLE 1: Test Set Distribution")
print("=" * 60)

# Collect test distribution data
table1_data = []

for result in all_results['per_device_local']:
    table1_data.append({
        'Device ID': result['device_num'],
        'Device Name': result['device_name'],
        'Cluster': result['cluster'],
        'Total Test Samples': result['n_test'],
        'Benign': result['n_test_benign'],
        'Malicious': result['n_test_malicious'],
        'Benign %': f"{(result['n_test_benign']/result['n_test']*100):.1f}%",
        'Malicious %': f"{(result['n_test_malicious']/result['n_test']*100):.1f}%"
    })

table1_df = pd.DataFrame(table1_data)

print("\nTABLE 1: Test Set Distribution per Device")
print("=" * 60)
print(table1_df.to_string(index=False))

# Save to CSV for LaTeX import
table1_df.to_csv(os.path.join(OUT_RESULTS, "table1_test_distribution.csv"), index=False)
print(f"\n✓ Saved to: {OUT_RESULTS}/table1_test_distribution.csv")

# Generate LaTeX table
latex_table1 = table1_df.to_latex(index=False, caption="Test Set Distribution per Device", 
                                    label="tab:test_distribution")
with open(os.path.join(OUT_RESULTS, "table1_test_distribution.tex"), 'w') as f:
    f.write(latex_table1)
print(f"✓ Saved LaTeX: {OUT_RESULTS}/table1_test_distribution.tex")

In [ ]:
# ============================================================
# TABLE 2: Comprehensive Per-Device Performance Comparison
# ============================================================

print("\n" + "=" * 60)
print("GENERATING TABLE 2: Per-Device Performance")
print("=" * 60)

# Collect per-device performance for all model types
table2_data = []

for i, local_res in enumerate(all_results['per_device_local']):
    cluster_res = all_results['per_device_cluster'][i]
    global_res = all_results['per_device_global'][i]
    
    table2_data.append({
        'Device ID': local_res['device_num'],
        'Device Name': local_res['device_name'][:25],  # Truncate for readability
        'Cluster': local_res['cluster'],
        # Local Model
        'Local Acc': f"{local_res['accuracy']:.4f}",
        'Local Prec': f"{local_res['precision']:.4f}",
        'Local Rec': f"{local_res['recall']:.4f}",
        'Local F1': f"{local_res['f1_score']:.4f}",
        # Cluster Model
        'Cluster Acc': f"{cluster_res['accuracy']:.4f}",
        'Cluster Prec': f"{cluster_res['precision']:.4f}",
        'Cluster Rec': f"{cluster_res['recall']:.4f}",
        'Cluster F1': f"{cluster_res['f1_score']:.4f}",
        # Global Model
        'Global Acc': f"{global_res['accuracy']:.4f}",
        'Global Prec': f"{global_res['precision']:.4f}",
        'Global Rec': f"{global_res['recall']:.4f}",
        'Global F1': f"{global_res['f1_score']:.4f}"
    })

table2_df = pd.DataFrame(table2_data)

print("\nTABLE 2: Comprehensive Per-Device Performance Comparison")
print("=" * 80)
print(table2_df.to_string(index=False))

# Save to CSV
table2_df.to_csv(os.path.join(OUT_RESULTS, "table2_per_device_performance.csv"), index=False)
print(f"\n✓ Saved to: {OUT_RESULTS}/table2_per_device_performance.csv")

# Generate LaTeX table
latex_table2 = table2_df.to_latex(index=False, 
                                   caption="Comprehensive Per-Device Performance Comparison",
                                   label="tab:per_device_performance")
with open(os.path.join(OUT_RESULTS, "table2_per_device_performance.tex"), 'w') as f:
    f.write(latex_table2)
print(f"✓ Saved LaTeX: {OUT_RESULTS}/table2_per_device_performance.tex")

In [ ]:
# ============================================================
# TABLE 3: Aggregate Performance Statistics
# ============================================================

print("\n" + "=" * 60)
print("GENERATING TABLE 3: Aggregate Statistics")
print("=" * 60)

# Compute aggregate statistics
def compute_aggregates(results_list, model_type):
    metrics = ['accuracy', 'precision', 'recall', 'f1_score']
    stats = {'Model Type': model_type}
    
    for metric in metrics:
        values = [r[metric] for r in results_list]
        stats[f'{metric.capitalize()} Mean'] = np.mean(values)
        stats[f'{metric.capitalize()} Std'] = np.std(values)
    
    return stats

table3_data = [
    compute_aggregates(all_results['per_device_local'], 'Local Model'),
    compute_aggregates(all_results['per_device_cluster'], 'Cluster Model'),
    compute_aggregates(all_results['per_device_global'], 'Global Model')
]

table3_df = pd.DataFrame(table3_data)

# Format for display
display_cols = ['Model Type', 'Accuracy_mean Mean', 'Accuracy_mean Std', 
                'Precision_mean Mean', 'Precision_mean Std',
                'Recall_mean Mean', 'Recall_mean Std',
                'F1_score_mean Mean', 'F1_score_mean Std']

# Rename columns for clarity
table3_display = table3_df.copy()
table3_display.columns = ['Model Type', 'Acc μ', 'Acc σ', 'Prec μ', 'Prec σ', 
                          'Rec μ', 'Rec σ', 'F1 μ', 'F1 σ']

# Format numbers
for col in table3_display.columns[1:]:
    table3_display[col] = table3_display[col].apply(lambda x: f"{x:.4f}")

print("\nTABLE 3: Aggregate Performance Statistics (Mean ± Std)")
print("=" * 80)
print(table3_display.to_string(index=False))

# Save to CSV
table3_display.to_csv(os.path.join(OUT_RESULTS, "table3_aggregate_statistics.csv"), index=False)
print(f"\n✓ Saved to: {OUT_RESULTS}/table3_aggregate_statistics.csv")

# Generate LaTeX table
latex_table3 = table3_display.to_latex(index=False,
                                       caption="Aggregate Performance Statistics Across All Devices",
                                       label="tab:aggregate_stats")
with open(os.path.join(OUT_RESULTS, "table3_aggregate_statistics.tex"), 'w') as f:
    f.write(latex_table3)
print(f"✓ Saved LaTeX: {OUT_RESULTS}/table3_aggregate_statistics.tex")

In [ ]:
# ============================================================
# TABLE 4: Aggregate Confusion Matrices
# ============================================================

print("\n" + "=" * 60)
print("GENERATING TABLE 4: Confusion Matrices")
print("=" * 60)

# Compute aggregate confusion matrices
def compute_aggregate_cm(results_list, model_type):
    total_tp = sum(r['tp'] for r in results_list)
    total_tn = sum(r['tn'] for r in results_list)
    total_fp = sum(r['fp'] for r in results_list)
    total_fn = sum(r['fn'] for r in results_list)
    
    total = total_tp + total_tn + total_fp + total_fn
    
    return {
        'Model Type': model_type,
        'True Negative (TN)': total_tn,
        'False Positive (FP)': total_fp,
        'False Negative (FN)': total_fn,
        'True Positive (TP)': total_tp,
        'TN %': f"{(total_tn/total*100):.2f}%",
        'FP %': f"{(total_fp/total*100):.2f}%",
        'FN %': f"{(total_fn/total*100):.2f}%",
        'TP %': f"{(total_tp/total*100):.2f}%"
    }

table4_data = [
    compute_aggregate_cm(all_results['per_device_local'], 'Local Model'),
    compute_aggregate_cm(all_results['per_device_cluster'], 'Cluster Model'),
    compute_aggregate_cm(all_results['per_device_global'], 'Global Model')
]

table4_df = pd.DataFrame(table4_data)

print("\nTABLE 4: Aggregate Confusion Matrices")
print("=" * 80)
print(table4_df.to_string(index=False))

# Save to CSV
table4_df.to_csv(os.path.join(OUT_RESULTS, "table4_confusion_matrices.csv"), index=False)
print(f"\n✓ Saved to: {OUT_RESULTS}/table4_confusion_matrices.csv")

# Generate LaTeX table
latex_table4 = table4_df.to_latex(index=False,
                                  caption="Aggregate Confusion Matrices Across All Devices",
                                  label="tab:confusion_matrices")
with open(os.path.join(OUT_RESULTS, "table4_confusion_matrices.tex"), 'w') as f:
    f.write(latex_table4)
print(f"✓ Saved LaTeX: {OUT_RESULTS}/table4_confusion_matrices.tex")

In [ ]:
# ============================================================
# FIGURE 1: Performance by Cluster
# ============================================================

print("\n" + "=" * 60)
print("GENERATING FIGURE 1: Performance by Cluster")
print("=" * 60)

import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

# Prepare data grouped by cluster
cluster_performance = {
    'Cluster 0': {'Local': [], 'Cluster': [], 'Global': []},
    'Cluster 1': {'Local': [], 'Cluster': [], 'Global': []}
}

for i, local_res in enumerate(all_results['per_device_local']):
    cluster_res = all_results['per_device_cluster'][i]
    global_res = all_results['per_device_global'][i]
    
    cluster_name = f"Cluster {local_res['cluster']}"
    cluster_performance[cluster_name]['Local'].append(local_res['f1_score'])
    cluster_performance[cluster_name]['Cluster'].append(cluster_res['f1_score'])
    cluster_performance[cluster_name]['Global'].append(global_res['f1_score'])

# Compute means per cluster
cluster_means = []
for cluster_name in ['Cluster 0', 'Cluster 1']:
    cluster_means.append({
        'Cluster': cluster_name,
        'Local': np.mean(cluster_performance[cluster_name]['Local']),
        'Cluster Model': np.mean(cluster_performance[cluster_name]['Cluster']),
        'Global': np.mean(cluster_performance[cluster_name]['Global']),
        'n_devices': len(cluster_performance[cluster_name]['Local'])
    })

cluster_means_df = pd.DataFrame(cluster_means)

# Create grouped bar chart
fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(cluster_means_df))
width = 0.25

bars1 = ax.bar(x - width, cluster_means_df['Local'], width, label='Local Model', 
               color='#3498db', alpha=0.8)
bars2 = ax.bar(x, cluster_means_df['Cluster Model'], width, label='Cluster Model',
               color='#2ecc71', alpha=0.8)
bars3 = ax.bar(x + width, cluster_means_df['Global'], width, label='Global Model',
               color='#e74c3c', alpha=0.8)

# Customize
ax.set_xlabel('Cluster', fontsize=12, fontweight='bold')
ax.set_ylabel('Mean F1-Score', fontsize=12, fontweight='bold')
ax.set_title('Model Performance Comparison by Cluster', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([f"{row['Cluster']}\n({row['n_devices']} devices)" 
                     for _, row in cluster_means_df.iterrows()])
ax.legend(loc='lower right', fontsize=11)
ax.set_ylim([0.999, 1.0001])
ax.grid(axis='y', alpha=0.3)

# Add value labels on bars
def add_value_labels(bars):
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.4f}', ha='center', va='bottom', fontsize=9)

add_value_labels(bars1)
add_value_labels(bars2)
add_value_labels(bars3)

plt.tight_layout()
plt.savefig(os.path.join(OUT_RESULTS, "figure1_performance_by_cluster.png"), 
            dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(OUT_RESULTS, "figure1_performance_by_cluster.pdf"), 
            bbox_inches='tight')
plt.show()

print(f"\n✓ Saved to: {OUT_RESULTS}/figure1_performance_by_cluster.png")
print(f"✓ Saved to: {OUT_RESULTS}/figure1_performance_by_cluster.pdf")

In [ ]:
# ============================================================
# FIGURE 2: ROC Curves for Model Comparison
# ============================================================

print("\n" + "=" * 60)
print("GENERATING FIGURE 2: ROC Curves")
print("=" * 60)

# Create ROC curves for representative devices (3 devices: one from each cluster + one more)
representative_devices = ['1', '4', '5']  # Danmini (Cluster 1), Baby Monitor (Cluster 0), Camera (Cluster 1)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for idx, device_num in enumerate(representative_devices):
    ax = axes[idx]
    
    # Get ROC data for this device
    device_roc_data = [r for r in all_results['roc_data'] if r['device_num'] == device_num]
    
    # Plot ROC curves for each model type
    colors = {'Local': '#3498db', 'Cluster': '#2ecc71', 'Global': '#e74c3c'}
    
    for roc_data in device_roc_data:
        model_type = roc_data['model_type']
        ax.plot(roc_data['fpr'], roc_data['tpr'], 
                color=colors[model_type], lw=2, alpha=0.8,
                label=f"{model_type} (AUC = {roc_data['auc']:.4f})")
    
    # Diagonal line
    ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.3, label='Random Classifier')
    
    # Customize
    device_name = DEVICE_NAME_MAP[device_num]
    ax.set_xlabel('False Positive Rate', fontsize=11)
    ax.set_ylabel('True Positive Rate', fontsize=11)
    ax.set_title(f'{device_name}\n(Device {device_num})', fontsize=12, fontweight='bold')
    ax.legend(loc='lower right', fontsize=9)
    ax.grid(alpha=0.3)
    ax.set_xlim([0.0, 1.0])
    ax.set_ylim([0.0, 1.05])

plt.suptitle('ROC Curves: Model Comparison Across Representative Devices', 
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(OUT_RESULTS, "figure2_roc_curves.png"), 
            dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(OUT_RESULTS, "figure2_roc_curves.pdf"), 
            bbox_inches='tight')
plt.show()

print(f"\n✓ Saved to: {OUT_RESULTS}/figure2_roc_curves.png")
print(f"✓ Saved to: {OUT_RESULTS}/figure2_roc_curves.pdf")

In [ ]:
# ============================================================
# FIGURE 3: F1-Score Comparison Across All Devices
# ============================================================

print("\n" + "=" * 60)
print("GENERATING FIGURE 3: F1-Score Comparison")
print("=" * 60)

# Prepare data
devices = [r['device_num'] for r in all_results['per_device_local']]
local_f1 = [r['f1_score'] for r in all_results['per_device_local']]
cluster_f1 = [r['f1_score'] for r in all_results['per_device_cluster']]
global_f1 = [r['f1_score'] for r in all_results['per_device_global']]

# Create figure
fig, ax = plt.subplots(figsize=(14, 6))

x = np.arange(len(devices))
width = 0.25

bars1 = ax.bar(x - width, local_f1, width, label='Local Model', 
               color='#3498db', alpha=0.8, edgecolor='black', linewidth=0.5)
bars2 = ax.bar(x, cluster_f1, width, label='Cluster Model',
               color='#2ecc71', alpha=0.8, edgecolor='black', linewidth=0.5)
bars3 = ax.bar(x + width, global_f1, width, label='Global Model',
               color='#e74c3c', alpha=0.8, edgecolor='black', linewidth=0.5)

# Customize
ax.set_xlabel('Device', fontsize=12, fontweight='bold')
ax.set_ylabel('F1-Score', fontsize=12, fontweight='bold')
ax.set_title('F1-Score Comparison Across All Devices and Model Types', 
             fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([f"D{d}" for d in devices], fontsize=10)
ax.legend(loc='lower right', fontsize=11)
ax.set_ylim([0.9995, 1.0001])
ax.grid(axis='y', alpha=0.3)

# Add horizontal line at 0.9999
ax.axhline(y=0.9999, color='gray', linestyle='--', linewidth=1, alpha=0.5)

plt.tight_layout()
plt.savefig(os.path.join(OUT_RESULTS, "figure3_f1_comparison.png"), 
            dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(OUT_RESULTS, "figure3_f1_comparison.pdf"), 
            bbox_inches='tight')
plt.show()

print(f"\n✓ Saved to: {OUT_RESULTS}/figure3_f1_comparison.png")
print(f"✓ Saved to: {OUT_RESULTS}/figure3_f1_comparison.pdf")

In [ ]:
# ============================================================
# FIGURE 4: Heatmap of Performance Metrics
# ============================================================

print("\n" + "=" * 60)
print("GENERATING FIGURE 4: Performance Heatmap")
print("=" * 60)

# Prepare data for heatmap (Global Model across all devices)
heatmap_data = []
for result in all_results['per_device_global']:
    heatmap_data.append({
        'Device': f"D{result['device_num']}",
        'Accuracy': result['accuracy'],
        'Precision': result['precision'],
        'Recall': result['recall'],
        'F1-Score': result['f1_score']
    })

heatmap_df = pd.DataFrame(heatmap_data)
heatmap_df = heatmap_df.set_index('Device')

# Create heatmap
fig, ax = plt.subplots(figsize=(10, 8))

sns.heatmap(heatmap_df.T, annot=True, fmt='.4f', cmap='RdYlGn', 
            vmin=0.995, vmax=1.0, cbar_kws={'label': 'Score'},
            linewidths=0.5, linecolor='gray', ax=ax)

ax.set_title('Global Model Performance Heatmap Across All Devices', 
             fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Device', fontsize=12, fontweight='bold')
ax.set_ylabel('Metric', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(OUT_RESULTS, "figure4_performance_heatmap.png"), 
            dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(OUT_RESULTS, "figure4_performance_heatmap.pdf"), 
            bbox_inches='tight')
plt.show()

print(f"\n✓ Saved to: {OUT_RESULTS}/figure4_performance_heatmap.png")
print(f"✓ Saved to: {OUT_RESULTS}/figure4_performance_heatmap.pdf")

In [ ]:
# ============================================================
# FINAL SUMMARY REPORT
# ============================================================

print("\n" + "=" * 60)
print("📊 SUMMARY: All Tables & Figures Generated")
print("=" * 60)

print("\n✅ TABLES:")
print("  1. table1_test_distribution.csv (.tex)")
print("  2. table2_per_device_performance.csv (.tex)")
print("  3. table3_aggregate_statistics.csv (.tex)")
print("  4. table4_confusion_matrices.csv (.tex)")

print("\n✅ FIGURES:")
print("  1. figure1_performance_by_cluster.png (.pdf)")
print("  2. figure2_roc_curves.png (.pdf)")
print("  3. figure3_f1_comparison.png (.pdf)")
print("  4. figure4_performance_heatmap.png (.pdf)")

print(f"\n📁 All files saved in: {OUT_RESULTS}/")
print("\n🎉 Ready to insert into your paper!")
print("=" * 60)

In [ ]:
# ============================================================
# AUTO-GENERATE LATEX TABLES WITH ACTUAL DATA
# ============================================================

print("\n" + "=" * 60)
print("GENERATING LATEX TABLES WITH ACTUAL DATA")
print("=" * 60)

# ---------- TABLE 1: Test Set Distribution ----------
print("\n📊 Generating LaTeX for Table 1...")

latex_table1 = r"""\begin{table}[htbp]
\centering
\caption{Test Set Distribution per Device}
\label{tab:test_distribution}
\begin{tabular}{llrrrrr}
\hline
\textbf{Device} & \textbf{Device Name} & \textbf{Cluster} & \textbf{Total} & \textbf{Benign} & \textbf{Malicious} & \textbf{Mal.\%} \\
\hline
"""

total_test = 0
total_benign = 0
total_malicious = 0

for result in all_results['per_device_local']:
    device_name_short = result['device_name'].replace('_', ' ')
    if len(device_name_short) > 25:
        device_name_short = device_name_short[:22] + "..."
    
    mal_pct = (result['n_test_malicious'] / result['n_test'] * 100)
    
    latex_table1 += f"{result['device_num']} & {device_name_short} & {result['cluster']} & "
    latex_table1 += f"{result['n_test']:,} & {result['n_test_benign']:,} & {result['n_test_malicious']:,} & "
    latex_table1 += f"{mal_pct:.1f}\\% \\\\\n"
    
    total_test += result['n_test']
    total_benign += result['n_test_benign']
    total_malicious += result['n_test_malicious']

latex_table1 += r"""\hline
"""
latex_table1 += f"\\textbf{{Total}} & & & \\textbf{{{total_test:,}}} & \\textbf{{{total_benign:,}}} & \\textbf{{{total_malicious:,}}} & \\textbf{{{(total_malicious/total_test*100):.1f}\\%}} \\\\\n"
latex_table1 += r"""\hline
\end{tabular}
\end{table}
"""

print(latex_table1)

with open(os.path.join(OUT_RESULTS, "latex_table1.tex"), 'w') as f:
    f.write(latex_table1)

print(f"\n✓ Saved to: {OUT_RESULTS}/latex_table1.tex")

# ---------- TABLE 2: Per-Device Performance (SIMPLIFIED) ----------
print("\n📊 Generating LaTeX for Table 2 (F1-Scores only)...")

latex_table2 = r"""\begin{table*}[htbp]
\centering
\caption{Per-Device F1-Score Comparison Across Model Types}
\label{tab:per_device_performance}
\begin{tabular}{llcccc}
\hline
\textbf{Device} & \textbf{Device Name} & \textbf{Cluster} & \textbf{Local F1} & \textbf{Cluster F1} & \textbf{Global F1} \\
\hline
"""

for i, local_res in enumerate(all_results['per_device_local']):
    cluster_res = all_results['per_device_cluster'][i]
    global_res = all_results['per_device_global'][i]
    
    device_name_short = local_res['device_name'].replace('_', ' ')
    if len(device_name_short) > 30:
        device_name_short = device_name_short[:27] + "..."
    
    latex_table2 += f"{local_res['device_num']} & {device_name_short} & {local_res['cluster']} & "
    latex_table2 += f"{local_res['f1_score']:.4f} & {cluster_res['f1_score']:.4f} & {global_res['f1_score']:.4f} \\\\\n"

latex_table2 += r"""\hline
\end{tabular}
\end{table*}
"""

print(latex_table2)

with open(os.path.join(OUT_RESULTS, "latex_table2.tex"), 'w') as f:
    f.write(latex_table2)

print(f"\n✓ Saved to: {OUT_RESULTS}/latex_table2.tex")

# ---------- TABLE 3: Aggregate Statistics ----------
print("\n📊 Generating LaTeX for Table 3...")

# Compute statistics
def compute_stats(results, metric):
    values = [r[metric] for r in results]
    return np.mean(values), np.std(values)

latex_table3 = r"""\begin{table}[htbp]
\centering
\caption{Aggregate Performance Statistics Across All Devices}
\label{tab:aggregate_stats}
\begin{tabular}{lcccc}
\hline
\textbf{Model Type} & \textbf{Accuracy} & \textbf{Precision} & \textbf{Recall} & \textbf{F1-Score} \\
\hline
"""

for model_name, results in [('Local Model', all_results['per_device_local']),
                             ('Cluster Model', all_results['per_device_cluster']),
                             ('Global Model', all_results['per_device_global'])]:
    
    acc_mean, acc_std = compute_stats(results, 'accuracy')
    prec_mean, prec_std = compute_stats(results, 'precision')
    rec_mean, rec_std = compute_stats(results, 'recall')
    f1_mean, f1_std = compute_stats(results, 'f1_score')
    
    latex_table3 += f"{model_name} & "
    latex_table3 += f"${acc_mean:.4f} \\pm {acc_std:.4f}$ & "
    latex_table3 += f"${prec_mean:.4f} \\pm {prec_std:.4f}$ & "
    latex_table3 += f"${rec_mean:.4f} \\pm {rec_std:.4f}$ & "
    latex_table3 += f"${f1_mean:.4f} \\pm {f1_std:.4f}$ \\\\\n"

latex_table3 += r"""\hline
\end{tabular}
\end{table}
"""

print(latex_table3)

with open(os.path.join(OUT_RESULTS, "latex_table3.tex"), 'w') as f:
    f.write(latex_table3)

print(f"\n✓ Saved to: {OUT_RESULTS}/latex_table3.tex")

# ---------- TABLE 4: Confusion Matrices ----------
print("\n📊 Generating LaTeX for Table 4...")

def compute_totals(results):
    total_tp = sum(r['tp'] for r in results)
    total_tn = sum(r['tn'] for r in results)
    total_fp = sum(r['fp'] for r in results)
    total_fn = sum(r['fn'] for r in results)
    total = total_tp + total_tn + total_fp + total_fn
    return total_tn, total_fp, total_fn, total_tp, total

latex_table4 = r"""\begin{table}[htbp]
\centering
\caption{Aggregate Confusion Matrices Across All Devices}
\label{tab:confusion_matrices}
\begin{tabular}{lrrrr}
\hline
\textbf{Model Type} & \textbf{TN} & \textbf{FP} & \textbf{FN} & \textbf{TP} \\
\hline
"""

for model_name, results in [('Local Model', all_results['per_device_local']),
                             ('Cluster Model', all_results['per_device_cluster']),
                             ('Global Model', all_results['per_device_global'])]:
    
    tn, fp, fn, tp, total = compute_totals(results)
    
    latex_table4 += f"{model_name} & "
    latex_table4 += f"{tn:,} ({tn/total*100:.2f}\\%) & "
    latex_table4 += f"{fp:,} ({fp/total*100:.2f}\\%) & "
    latex_table4 += f"{fn:,} ({fn/total*100:.2f}\\%) & "
    latex_table4 += f"{tp:,} ({tp/total*100:.2f}\\%) \\\\\n"

latex_table4 += r"""\hline
\end{tabular}
\end{table}
"""

print(latex_table4)

with open(os.path.join(OUT_RESULTS, "latex_table4.tex"), 'w') as f:
    f.write(latex_table4)

print(f"\n✓ Saved to: {OUT_RESULTS}/latex_table4.tex")

# ---------- SUMMARY ----------
print("\n" + "=" * 60)
print("✅ ALL LATEX TABLES GENERATED WITH ACTUAL DATA")
print("=" * 60)
print("\nGenerated files:")
print("  - latex_table1.tex (Test Distribution)")
print("  - latex_table2.tex (Per-Device Performance)")
print("  - latex_table3.tex (Aggregate Statistics)")
print("  - latex_table4.tex (Confusion Matrices)")
print(f"\nLocation: {OUT_RESULTS}/")
print("\n🎯 Download these .tex files and copy-paste into your paper!")
print("=" * 60)

In [ ]:
# ============================================================
# FIGURE 3: Average F1-Score per Model Type
# ============================================================

print("\n" + "=" * 60)
print("GENERATING FIGURE 3: Average F1-Score per Model Type")
print("=" * 60)

# ---------- Extract F1 scores ----------
local_f1_scores = [
    r["f1_score"] for r in all_results["per_device_local"]
    if r.get("f1_score") is not None
]

cluster_f1_scores = [
    r["f1_score"] for r in all_results["per_device_cluster"]
    if r.get("f1_score") is not None
]

global_f1_scores = [
    r["f1_score"] for r in all_results["per_device_global"]
    if r.get("f1_score") is not None
]

# ---------- Sanity check ----------
print(f"Local models:   {len(local_f1_scores)} devices")
print(f"Cluster models: {len(cluster_f1_scores)} devices")
print(f"Global models:  {len(global_f1_scores)} devices")

# ---------- Compute averages ----------
avg_f1 = [
    np.mean(local_f1_scores),
    np.mean(cluster_f1_scores),
    np.mean(global_f1_scores)
]

model_labels = ["Local Models", "Cluster Models", "Global Model"]

# ---------- Plot ----------
fig, ax = plt.subplots(figsize=(8, 5))

bars = ax.bar(
    model_labels,
    avg_f1,
    alpha=0.85,
    edgecolor="black",
    linewidth=0.8
)

ax.set_ylabel("Average F1-Score", fontsize=12, fontweight="bold")
ax.set_title("Average F1-Score Comparison Across Model Types",
             fontsize=14, fontweight="bold")
ax.set_ylim(0.95, 1.001)
ax.grid(axis="y", alpha=0.3)

# ---------- Annotate ----------
for bar in bars:
    height = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        height,
        f"{height:.4f}",
        ha="center",
        va="bottom",
        fontsize=10,
        fontweight="bold"
    )

plt.tight_layout()
plt.savefig(os.path.join(OUT_RESULTS, "figure3_avg_f1_model_comparison.png"),
            dpi=300, bbox_inches="tight")
plt.savefig(os.path.join(OUT_RESULTS, "figure3_avg_f1_model_comparison.pdf"),
            bbox_inches="tight")
plt.show()

print(f"\n✓ Saved to: {OUT_RESULTS}/figure3_avg_f1_model_comparison.png")
print(f"✓ Saved to: {OUT_RESULTS}/figure3_avg_f1_model_comparison.pdf")


In [ ]:
print(all_results.keys())
print("Cluster F1s:", cluster_f1_scores)
